# Démo 3 — Réponses et sources

**Contexte :** assistant du service client de Lagune Banque.

Cette démo compare deux réponses à une même question : l'une sans documentation, l'autre avec les documents de la banque. Le passage source permet de vérifier les affirmations.

**Sommaire**
- [Configuration](#configuration)
- [Documentation et comparaison](#documentation)
- [À retenir](#a-retenir)

## <a id="configuration"></a>Configuration

In [1]:
import os
import sys
import time
from pathlib import Path

import litellm

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../../../code_partage")] if (p / "config.py").exists()).resolve()))
from config import prepare
import rag

In [2]:
prepare()

Connexion OK : gemini/gemini-3.5-flash-lite


In [3]:
SEED = 42


def ask(messages, **options):
    debut = time.perf_counter()
    r = litellm.completion(model=options.pop("model", os.environ["MODELE"]), api_key=os.environ["LLM_API_KEY"], seed=SEED, num_retries=4, messages=messages, **options)
    return {"text": (r.choices[0].message.content or "").strip(), "in": r.usage.prompt_tokens, "out": r.usage.completion_tokens, "seconds": time.perf_counter() - debut, "raw": r}

## <a id="documentation"></a>Documentation et comparaison

`DOCUMENTATION` rassemble les dix documents dans une chaîne de texte brut. L'aperçu montre ses deux premières lignes ; l'essai avec documents transmet le texte complet au modèle. Ce n'est pas encore du RAG.

La question porte sur une information présente dans les documents. Le modèle y répond une première fois sans documentation, puis une seconde fois avec `DOCUMENTATION`. Compare les deux réponses à la source affichée.

In [4]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
DOCUMENTATION = "\n\n".join(f"[{d['id']} : {d['titre']}]\n" + "\n".join(texte for _, texte in d["pages"]) for d in documents)

nb_tokens = litellm.token_counter(model=os.environ["MODELE"], text=DOCUMENTATION)
display_documentation = "\n".join(DOCUMENTATION.splitlines()[:5])
tokens_formates = f"{nb_tokens:,}".replace(",", " ")
from html import escape
from IPython.display import HTML, display

display(HTML(f"""
<div style="font-family:Arial,sans-serif;max-width:800px;color:#202124">
  <strong><code>DOCUMENTATION</code> · ses deux premières lignes</strong>
  <pre style="margin:8px 0;padding:12px 14px;white-space:pre-wrap;overflow-wrap:anywhere;background:#f6f8fa;border:1px solid #d0d7de;border-radius:6px;color:#24292f;font:13px/1.5 Consolas,monospace">{escape(display_documentation)}</pre>
  <span style="color:#57606a;font-size:12px">Le texte complet, soit {tokens_formates} tokens, sera ajouté au prompt dans les essais concernés.</span>
</div>
"""))

In [5]:
ROLE = """
    Tu es l'assistant du service client de Lagune Banque.
    Réponds en deux phrases au plus.
"""

SYSTEME_DOCUMENTS = f"{ROLE}\n\nDocumentation :\n{DOCUMENTATION}"

QUESTION_DOCUMENTEE = """
Quels frais s'appliquent si une échéance du Prêt Express+ est impayée ?
"""

reponses = {
    "documentee_sans_documents": ask([
        {"role": "system", "content": ROLE},
        {"role": "user", "content": QUESTION_DOCUMENTEE},
    ], max_tokens=200),
    "documentee_avec_documents": ask([
        {"role": "system", "content": SYSTEME_DOCUMENTS},
        {"role": "user", "content": QUESTION_DOCUMENTEE},
    ], max_tokens=200),
}

In [6]:
def carte_reponse(titre, reponse):
    texte = escape(reponse["text"]).replace("\n", "<br>")
    tokens_entree = f"{reponse['in']:,}".replace(",", " ")
    return f"<div style='box-sizing:border-box;height:100%;padding:12px 14px;background:#fff;border:1px solid #d0d7de;border-radius:7px'><div style='font-size:12px;font-weight:700;color:#57606a'>{escape(titre)}</div><p style='margin:8px 0;line-height:1.55;color:#202124'>{texte}</p><div style='font-size:11px;color:#667085'>{tokens_entree} tokens d'entrée</div></div>"

display(HTML(f"""
<div style="font-family:Arial,sans-serif;max-width:860px;color:#202124">
  <div style="margin:0 0 6px;font-size:16px;font-weight:700">Une réponse vérifiable</div>
  <div style="margin-bottom:10px;color:#475467">{escape(QUESTION_DOCUMENTEE)}</div>
  <div style="display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px">
    {carte_reponse('SANS DOCUMENTATION', reponses['documentee_sans_documents'])}
    {carte_reponse('AVEC DOCUMENTATION', reponses['documentee_avec_documents'])}
  </div>
</div>
"""))

## <a id="a-retenir"></a>À retenir

- Sans source, le modèle peut formuler une réponse plausible sans élément permettant de la vérifier.
- Avec une source dans le prompt, les affirmations peuvent être comparées au document.
- Cette comparaison illustre un cas précis ; elle ne permet pas de généraliser à tous les modèles.
- Envoyer tous les documents à chaque question augmente le coût ; le module 2 montrera comment ne récupérer que les passages utiles (RAG).